# Consolidate the five extensions
Attach the private input made from MSLD_Final_CPU_Inputs.zip. Set the accelerator to None and run the notebook.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, time, zipfile
STARTED = time.time()
EXPECTED = '72ca98e5bd5f5b6d1e64ca56786bb6513cdda952c21057d89bac948f7fd27fa6'
KAGGLE = Path('/kaggle/input').is_dir()
SEARCH = Path('/kaggle/input') if KAGGLE else Path.cwd()
WORK = Path('/kaggle/working') if KAGGLE else Path.cwd()
WORK.mkdir(parents=True,exist_ok=True)
def digest(p):
    with p.open('rb') as s:
        h=hashlib.sha256()
        for b in iter(lambda:s.read(2**20),b''):h.update(b)
        return h.hexdigest()
def verify(folder):
    manifest=folder/'extension_manifest.json'
    if not manifest.is_file() or digest(manifest)!=EXPECTED:return False
    for name,h in json.loads(manifest.read_text()).items():
        file=(folder/name).resolve()
        if not file.is_relative_to(folder.resolve()) or not file.is_file() or digest(file)!=h:
            raise ValueError('Missing or changed frozen file: '+name)
    return True
def locate():
    for p in SEARCH.rglob('extension_manifest.json'):
        if verify(p.parent):return p.parent
    for pattern in ('MSLD_Extension_Inputs.zip','MSLD_Extension_Inputs.bin'):
        for archive in SEARCH.rglob(pattern):
            target=Path(tempfile.mkdtemp(prefix='msld_extension_'))
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    dest=(target/member.filename).resolve()
                    if not dest.is_relative_to(target.resolve()):raise ValueError('Invalid archive path')
                    if member.is_dir():continue
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(member) as src,dest.open('wb') as dst:shutil.copyfileobj(src,dst)
            if verify(target):return target
    raise FileNotFoundError('Attach MSLD_Extension_Inputs.zip or place it beside the notebook.')
# Kaggle can mount the single upload as a ZIP or expand it into directories.
outer=list(SEARCH.rglob('MSLD_Final_CPU_Inputs.zip'))
if len(outer)>1:raise ValueError('Attach just one version of the final CPU input dataset.')
if outer:
    if digest(outer[0])!='db71b8f5d2d2c9d374c59c3ce76e00be3a3cf5ad845fbeeaaa419afaa081d75f':raise ValueError('The single-upload bundle has changed.')
    unpacked=Path(tempfile.mkdtemp(prefix='msld_final_cpu_'))
    with zipfile.ZipFile(outer[0]) as bundle:
        for member in bundle.infolist():
            dest=(unpacked/member.filename).resolve()
            if not dest.is_relative_to(unpacked.resolve()):raise ValueError(member.filename)
            dest.parent.mkdir(parents=True,exist_ok=True)
            with bundle.open(member) as src,dest.open('wb') as dst:shutil.copyfileobj(src,dst)
    SEARCH=unpacked
PACKAGE=locate()
print('Frozen inputs verified.',EXPECTED)

def restore(owner):
    destination=WORK/('EXT_'+owner)
    prior=[]
    for p in SEARCH.rglob('input_identity.json'):
        record=json.loads(p.read_text())
        if p.parent.resolve()!=destination.resolve() and record.get('worker')==destination.name and record.get('input_manifest_sha256')==EXPECTED:
            prior.append(p.parent)
    if len(prior)>1:raise ValueError('Attach only this worker’s latest full output version.')
    if prior and not destination.exists():
        index=prior[0]/'result_index.json'
        if index.is_file():
            for rel,h in json.loads(index.read_text())['files'].items():
                source=(prior[0]/rel).resolve()
                if not source.is_relative_to(prior[0].resolve()) or not source.is_file() or digest(source)!=h:
                    raise ValueError('Incomplete or changed previous output: '+rel)
        shutil.copytree(prior[0],destination)
    destination.mkdir(exist_ok=True)
    return destination



In [ ]:
import importlib.util
needed={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn','PIL':'Pillow','joblib':'joblib','threadpoolctl':'threadpoolctl'}
missing=[pkg for mod,pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)


In [ ]:
import sys, tempfile
sys.path.insert(0,str(PACKAGE/'code'))
from consolidate import main
archives={}
bundle_root=Path(tempfile.mkdtemp(prefix='ext_result_inputs_'))
for key in ('CPU','Me','Friend1','Friend2'):
    name=f'EXT_{key}_RESULTS_ONLY.zip'
    found=list(SEARCH.rglob(name))
    if len(found)>1:raise ValueError('Attach only the final result for '+key)
    if found:archives[key]=found[0];continue
    # Kaggle may have expanded a results ZIP into a separate dataset directory.
    matches=[]
    for p in SEARCH.rglob('result_index.json'):
        idx=json.loads(p.read_text())
        if idx.get('worker')==key and idx.get('complete') and (p.parent/'input_identity.json').is_file():matches.append(p)
    if len(matches)!=1:raise FileNotFoundError(f'Expected one completed source for {key}, found {len(matches)}')
    p=matches[0];dest=bundle_root/name
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        z.write(p,'result_index.json')
        for rel in json.loads(p.read_text())['files']:
            source=(p.parent/rel).resolve()
            if not source.is_relative_to(p.parent.resolve()):raise ValueError(rel)
            z.write(source,rel)
    archives[key]=dest
target=WORK/'EXT_Publication'
main(PACKAGE,archives,target)
print(shutil.make_archive(str(WORK/'EXT_Publication_Tables'),'zip',target))
